In [1]:
import pickle

In [4]:
def find_vocab(vocab_path, n = 40):
    with open(vocab_path, "rb") as f:
        vocab = pickle.load(f)
    
    print("type:", type(vocab))
    print("size:", len(vocab))
    
    print("\nFirst tokens:")
    for token_id in sorted(vocab.keys())[:n]:
        token_bytes = vocab[token_id]
        print(f"{token_id:>5} | {token_bytes!r} | {token_bytes.decode('utf-8', errors='replace')!r}")
    
    print("\nLast tokens:")
    for token_id in sorted(vocab.keys())[-n:]:
        token_bytes = vocab[token_id]
        print(f"{token_id:>5} | {token_bytes!r} | {token_bytes.decode('utf-8', errors='replace')!r}")
    
    longest_id = max(vocab.keys(), key=lambda i: len(vocab[i]))
    longest_token = vocab[longest_id]
    print("\nLongest token:")
    print("id:", longest_id)
    print("length:", len(longest_token))
    print("bytes:", longest_token)
    print("decoded:", longest_token.decode("utf-8", errors="replace"))

In [5]:
find_vocab("/home/ygm/llm-project/CS336-From-Scratch/Assignment/data/tinystories_vocab.pkl")

type: <class 'dict'>
size: 10000

First tokens:
    0 | b'\x00' | '\x00'
    1 | b'\x01' | '\x01'
    2 | b'\x02' | '\x02'
    3 | b'\x03' | '\x03'
    4 | b'\x04' | '\x04'
    5 | b'\x05' | '\x05'
    6 | b'\x06' | '\x06'
    7 | b'\x07' | '\x07'
    8 | b'\x08' | '\x08'
    9 | b'\t' | '\t'
   10 | b'\n' | '\n'
   11 | b'\x0b' | '\x0b'
   12 | b'\x0c' | '\x0c'
   13 | b'\r' | '\r'
   14 | b'\x0e' | '\x0e'
   15 | b'\x0f' | '\x0f'
   16 | b'\x10' | '\x10'
   17 | b'\x11' | '\x11'
   18 | b'\x12' | '\x12'
   19 | b'\x13' | '\x13'
   20 | b'\x14' | '\x14'
   21 | b'\x15' | '\x15'
   22 | b'\x16' | '\x16'
   23 | b'\x17' | '\x17'
   24 | b'\x18' | '\x18'
   25 | b'\x19' | '\x19'
   26 | b'\x1a' | '\x1a'
   27 | b'\x1b' | '\x1b'
   28 | b'\x1c' | '\x1c'
   29 | b'\x1d' | '\x1d'
   30 | b'\x1e' | '\x1e'
   31 | b'\x1f' | '\x1f'
   32 | b' ' | ' '
   33 | b'!' | '!'
   34 | b'"' | '"'
   35 | b'#' | '#'
   36 | b'$' | '$'
   37 | b'%' | '%'
   38 | b'&' | '&'
   39 | b"'" | "'"

Last tokens

In [7]:
str = "the cat ate"

In [8]:
# handout 里的 vocab
vocab = {
    0: b" ",
    1: b"a",
    2: b"c",
    3: b"e",
    4: b"h",
    5: b"t",
    6: b"th",
    7: b" c",
    8: b" a",
    9: b"the",
    10: b" at",
}

# handout 里的 merges，按创建顺序排列
merges = [
    (b"t", b"h"),
    (b" ", b"c"),
    (b" ", b"a"),
    (b"th", b"e"),
    (b" a", b"t"),
]

print(vocab)
print(merges)

{0: b' ', 1: b'a', 2: b'c', 3: b'e', 4: b'h', 5: b't', 6: b'th', 7: b' c', 8: b' a', 9: b'the', 10: b' at'}
[(b't', b'h'), (b' ', b'c'), (b' ', b'a'), (b'th', b'e'), (b' a', b't')]


In [9]:
token_to_id = {token_bytes: token_id for token_id,token_bytes in vocab.items()}

In [10]:
token_to_id

{b' ': 0,
 b'a': 1,
 b'c': 2,
 b'e': 3,
 b'h': 4,
 b't': 5,
 b'th': 6,
 b' c': 7,
 b' a': 8,
 b'the': 9,
 b' at': 10}

In [11]:
merge_ranks = {pair: rank for rank,pair in enumerate(merges)}

In [12]:
merge_ranks

{(b't', b'h'): 0,
 (b' ', b'c'): 1,
 (b' ', b'a'): 2,
 (b'th', b'e'): 3,
 (b' a', b't'): 4}

In [13]:
import regex as re

In [14]:
GPT2_PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
compiled_pat = re.compile(GPT2_PAT)

pretokens = [m.group(0) for m in compiled_pat.finditer(str)]
pretokens

['the', ' cat', ' ate']

In [15]:
def pretoken_to_byte_token(pretoken: str):
    return [bytes([b]) for b in pretoken.encode("utf-8")]

In [16]:
for p in pretokens:
    print(p, "->", pretoken_to_byte_token(p))

the -> [b't', b'h', b'e']
 cat -> [b' ', b'c', b'a', b't']
 ate -> [b' ', b'a', b't', b'e']


In [19]:
def get_merge(tokens: list[bytes], merge_ranks: dict[tuple[bytes, bytes], int]):
    candidates = []
    for i in range(len(tokens) - 1):
        pair = (tokens[i], tokens[i + 1])
        if pair in merge_ranks:
            candidates.append((merge_ranks[pair], i, pair))
    return candidates

In [20]:
def merge_once(tokens: list[bytes], merge_ranks: dict[tuple[bytes, bytes], int]):
    candidates = get_merge(tokens,merge_ranks)
    if not candidates:
        return tokens, False
    
    _,i,pair = min(candidates)
    merge_token = pair[0] + pair[1]
    new_token = tokens[:i] + [merge_token] + tokens[i+2:]
    return new_token, True

In [22]:
def encode_pretoken_to_tokens(pretoken: str, merge_ranks: dict[tuple[bytes, bytes], int]):
    tokens = pretoken_to_byte_token(pretoken)
    
    while True:
        tokens, changed = merge_once(tokens,merge_ranks)
        if not changed:
            break
    return tokens

In [23]:
for p in pretokens:
    final_tokens = encode_pretoken_to_tokens(p, merge_ranks)
    print(p, "->", final_tokens)

the -> [b'the']
 cat -> [b' c', b'a', b't']
 ate -> [b' at', b'e']


In [24]:
def tokens_to_ids(tokens: list[bytes], token_to_id: dict[bytes, int]):
    return [token_to_id[tok] for tok in tokens]

In [25]:
for p in pretokens:
    final_tokens = encode_pretoken_to_tokens(p, merge_ranks)
    ids = tokens_to_ids(final_tokens, token_to_id)
    print(p, "->", ids)

the -> [9]
 cat -> [7, 1, 5]
 ate -> [10, 3]


In [27]:
def encode(text: str):
    pretokens = [m.group(0) for m in compiled_pat.finditer(text)]
    all_ids = []
    
    for pretoken in pretokens:
        final_tokens = encode_pretoken_to_tokens(pretoken,merge_ranks)
        all_ids.extend(tokens_to_ids(final_tokens,token_to_id))
    return all_ids

In [28]:
encode_ids = encode("the cat ate")

In [29]:
encode_ids

[9, 7, 1, 5, 10, 3]

In [30]:
def decode(ids: list[int]):
    merge_bytes = b"".join(vocab[i] for i in ids)
    return merge_bytes.decode("utf-8", errors="replace")

In [31]:
text = decode(encode_ids)

In [32]:
text

'the cat ate'

In [33]:
special_tokens = ["<|endoftext|>"]

# 假设 special token 已经在 vocab 里
special_vocab = dict(vocab)
special_token_id = max(special_vocab.keys()) + 1
special_vocab[special_token_id] = b"<|endoftext|>"

In [39]:
special_vocab

{0: b' ',
 1: b'a',
 2: b'c',
 3: b'e',
 4: b'h',
 5: b't',
 6: b'th',
 7: b' c',
 8: b' a',
 9: b'the',
 10: b' at',
 11: b'<|endoftext|>'}

In [34]:
special_token_to_id = {token_bytes: token_id for token_id, token_bytes in special_vocab.items()}
pattern = re.compile("(" + "|".join(re.escape(tok) for tok in special_tokens) + ")")

In [40]:
special_token_to_id

{b' ': 0,
 b'a': 1,
 b'c': 2,
 b'e': 3,
 b'h': 4,
 b't': 5,
 b'th': 6,
 b' c': 7,
 b' a': 8,
 b'the': 9,
 b' at': 10,
 b'<|endoftext|>': 11}

In [36]:
text2 = "the<|endoftext|> cat"

In [37]:
parts = [p for p in pattern.split(text2) if p]

In [38]:
parts

['the', '<|endoftext|>', ' cat']

In [41]:
def encode_with_special(text: str) -> list[int]:
    parts = [p for p in pattern.split(text) if p]
    all_ids = []

    for part in parts:
        if part in special_tokens:
            all_ids.append(special_token_to_id[part.encode("utf-8")])
        else:
            pretokens = [m.group(0) for m in compiled_pat.finditer(part)]
            for pretoken in pretokens:
                final_tokens = encode_pretoken_to_tokens(pretoken, merge_ranks)
                all_ids.extend(tokens_to_ids(final_tokens, special_token_to_id))

    return all_ids

In [43]:
ids = encode_with_special(text2)

In [44]:
ids

[9, 11, 7, 1, 5]

In [46]:
def decode_with_special(ids: list[int]):
    merge_bytes = b"".join(special_vocab[i] for i in ids)
    return merge_bytes.decode("utf-8", errors="replace")

In [47]:
text3 = decode_with_special(ids)

In [48]:
text3

'the<|endoftext|> cat'